# rank

> Code-aware reranking on top of RRF: identifier stems, symbol-query detection, and post-merge scoring signals (inspired by semble, but using the extra structure kosha stores: pagerank, public_api, definitions).

In [ ]:
#| default_exp rank

In [ ]:
#| export
import re, math
from collections import Counter
from fastcore.all import L

## Identifier stems

> FTS tokenizers split `snake_case` (the `_` is a separator) but keep `camelCase` as one token. These helpers split identifiers into stems so both queries and indexed chunks match at the subtoken level, like semble's "identifier stems" signal: querying `parse config` should find `parseConfig`, `ConfigParser`, and `config_parser`.

In [ ]:
#| export
_camel_re = re.compile(r'(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])')
_tok_re   = re.compile(r'[A-Za-z_][A-Za-z0-9_]*')

def split_ident(tok:str) -> list:
	'Split an identifier into lowercase stems: getUserById -> [get,user,by,id]; parse_config -> [parse,config].'
	parts = [p for seg in tok.replace('::','.').split('.') for s in seg.split('_') for p in _camel_re.split(s) if p]
	return [p.lower() for p in parts]

def is_symbol(tok:str) -> bool:
	'True if tok looks like a code symbol: snake_case, camelCase, dotted, or :: qualified.'
	t = tok.strip('.')
	return '_' in t or '.' in t or '::' in t or bool(_camel_re.search(t))

def q_stems(q:str) -> set:
	'All identifier stems in a query string.'
	return {s for t in _tok_re.findall(q) for s in split_ident(t)}

def is_symbol_query(q:str) -> bool:
	'True if any query token is symbol-like, meaning lexical signals should dominate ranking.'
	return any(is_symbol(t) for t in _tok_re.findall(q))

def expand_query(q:str) -> str:
	'''Replace symbol-like tokens with their stems so FTS (implicit AND between terms) matches subtokens:
	"parseConfig save" -> "parse config save". Plain words pass through untouched.'''
	def ex(t):
		if not is_symbol(t): return t
		return ' '.join(dict.fromkeys(s for s in split_ident(t) if len(s) > 1)) or t
	return ' '.join(ex(t) for t in q.split()) or q

def chunk_stems(meta:dict) -> str:
	'Space-joined identifier stems of a chunk name + module path. Stored in metadata so FTS matches subtokens at index time.'
	toks = f"{meta.get('name') or ''} {meta.get('mod_name') or ''}"
	return ' '.join(dict.fromkeys(s for t in _tok_re.findall(toks) for s in split_ident(t) if len(s) > 1))

In [ ]:
assert split_ident('getUserById') == ['get','user','by','id']
assert split_ident('parse_config') == ['parse','config']
assert split_ident('HTTPServerError') == ['http','server','error']
assert split_ident('fastcore.basics.merge') == ['fastcore','basics','merge']
assert is_symbol('rrf_merge') and is_symbol('parseConfig') and is_symbol('Foo::bar')
assert not is_symbol('search') and not is_symbol('atomic')
assert is_symbol_query('where is parseConfig defined') and not is_symbol_query('atomic write temp file')
assert expand_query('parseConfig save') == 'parse config save'
assert expand_query('updateRepo') == 'update repo'
assert expand_query('atomic write') == 'atomic write'
assert chunk_stems(dict(name='vec_search', mod_name='litesearch.core.vec_search')) == 'vec search litesearch core'


## Rerank

> Reciprocal Rank Fusion treats every store the same; this pass re-orders the merged list with code-aware signals. Unlike semble we also have a call graph, so pagerank and public_api join the usual definition/stem/coherence/noise signals.

In [ ]:
#| export
_noise_re = re.compile(
	r'(^|/)(tests?|testing|examples?|samples?|benchmarks?|conftest|legacy|compat|deprecated|_vendor|vendored)([/_.]|$)'
	r'|\.d\.ts$|(^|/)test_[^/]*$|_test\.[^/.]+$', re.I)
_def_types = frozenset({'FunctionDef','AsyncFunctionDef','ClassDef'})

def _signals(r:dict, qs:set, sym:bool, max_pr:float, paths:Counter) -> dict:
	'Per-result code-aware ranking signals, each normalized to roughly [0,1].'
	m = r.get('metadata') or {}
	name, mod, typ = m.get('name') or '', m.get('mod_name') or '', m.get('type') or ''
	st = set(split_ident(name)) | (set(split_ident(mod)) if mod else set())
	ov = len(qs & st)/len(qs) if qs and st else 0.0
	p = str(m.get('path') or r.get('path') or '')
	return dict(
		stem  = ov * (1.5 if sym else 1.0),                                  # identifier stem overlap with query
		defn  = 1.0 if typ in _def_types and ov >= 0.5 else 0.0,             # chunk defines the queried symbol
		pr    = (r.get('pagerank') or 0.0)/max_pr,                           # call-graph centrality (blast radius)
		pub   = 1.0 if m.get('public_api') else 0.0,                         # exported/public symbol
		coh   = math.log2(paths[p]) if p and paths[p] > 1 else 0.0,          # multiple hits from the same file
		noise = 1.0 if p and _noise_re.search(p) else 0.0,                   # tests/examples/legacy shims
		doc   = 1.0 if m.get('docstring') else 0.0)                          # documented chunk

_prior_keys = frozenset({'pr','pub','coh','doc'})

def rerank(results, q:str, weights:dict=None, prior_scale:float=0.25) -> L:
	"""Code-aware rerank of RRF-merged results. Works in rank space: a signal with weight w moves a
	result up by ~w positions, so ordering is scale-free and ties keep their RRF order. Signals:
	definition-of-queried-symbol, identifier stem overlap, pagerank, public_api, same-file coherence,
	noise penalty (tests/examples/shims), docstring presence. Adaptive like semble: symbol-like
	queries get the full prior signals (pagerank/public/coherence/doc); natural-language queries
	scale them by prior_scale so an already-good RRF ordering isn't churned."""
	res = list(results)
	if len(res) < 2: return L(res)
	w = dict(defn=8, stem=6, pr=5, pub=2, coh=2, noise=-8, doc=1) | dict(weights or {})
	qs, sym = q_stems(q), is_symbol_query(q)
	pri = 1.0 if sym else prior_scale
	max_pr = max((r.get('pagerank') or 0.0) for r in res) or 1.0
	_p = lambda r: str((r.get('metadata') or {}).get('path') or r.get('path') or '')
	paths = Counter(_p(r) for r in res)
	def adj(ir):
		i, r = ir
		s = _signals(r, qs, sym, max_pr, paths)
		return i - sum(w[k]*v*(pri if k in _prior_keys else 1.0) for k, v in s.items())
	return L([r for _, r in sorted(enumerate(res), key=adj)])

In [ ]:
mk = lambda name, typ='FunctionDef', path='/repo/a.py', pr=0, pub=False, doc=None: dict(
    content=f'def {name}(): ...', pagerank=pr,
    metadata=dict(name=name, mod_name=f'pkg.mod.{name}', type=typ, path=path, public_api=pub, docstring=doc))

# definition of the queried symbol beats an unrelated earlier hit
rs = [mk('helper'), mk('other'), mk('rrf_merge', pub=True, doc='merge results')]
out = rerank(rs, 'rrf_merge')
assert out[0]['metadata']['name'] == 'rrf_merge'

# noise penalty pushes test files below equally-matched non-test hits
rs = [mk('save_model', path='/repo/tests/test_save.py'), mk('save_model', path='/repo/core.py'),
      mk('save_model', path='/repo/io.py')]
out = rerank(rs, 'save model')
assert out[-1]['metadata']['path'] == '/repo/tests/test_save.py'

# pagerank breaks near-ties (full priors on symbol-like queries)
rs = [mk('a'), mk('b', pr=0.9), mk('c')]
assert rerank(rs, 'unrelated_query zzz')[0]['metadata']['name'] == 'b'
# natural-language query with a confident RRF winner: priors are scaled down, order kept
assert rerank(rs, 'unrelated query zzz', prior_scale=0.0)[0]['metadata']['name'] == 'a'

# stable for empty/singleton
assert list(rerank([], 'q')) == []
